In [3]:
import pandas as pd
from datetime import datetime as dt

In [124]:
import os 
print(os.getcwd())

import glob

all_csv = glob.glob("*_sentiment.csv")
print(all_csv)

C:\Users\User\OneDrive - Singapore Management University\Year 4\IS453 - Financial Analytics\Project\Stocksense\Sentiment Calculator
['ATER_cleaned_sentiment.csv', 'cleaned_AAPL_sentiment.csv']


In [125]:
def group_by_date(data):
    data.columns= data.columns.str.lower()
    if 'datetime' in data.columns:
        data['date'] = pd.to_datetime(data['datetime']).dt.date
    else:
        data['date'] = pd.to_datetime(data['date'])
    return data.groupby('date').mean()


In [126]:
def weighted_weekends(data):
    weighted_flair = weighted_vader = weighted_finbert = 0
    new_df = []
    for index, row in data.iterrows():
        
        day = index.weekday()
        if (day == 5): # saturday
            weighted_flair += row['flair_sentiment_score'] * 0.2
            weighted_vader += row['vader_score'] * 0.2
            weighted_finbert += row['finbert_score'] * 0.2
        elif (day == 6): # sunday
            weighted_flair += row['flair_sentiment_score'] * 0.3
            weighted_vader += row['vader_score'] * 0.3
            weighted_finbert += row['finbert_score'] * 0.3
        elif (day == 0): # monday
            weighted_flair += row['flair_sentiment_score'] * 0.5
            weighted_vader += row['vader_score'] * 0.5
            weighted_finbert += row['finbert_score'] * 0.5
            new_df.append([index, weighted_flair, weighted_vader, weighted_finbert])
            weighted_flair = weighted_vader = weighted_finbert = 0
        else:
            new_df.append([index, row['flair_sentiment_score'], row['vader_score'], row['finbert_score']])
            
    return new_df

In [127]:
for file in all_csv:
    file_df = pd.read_csv(file)
    data = group_by_date(file_df)
    weighted_sent = pd.DataFrame(weighted_weekends(data), columns = ['date', 'flair_weighted', 'vader_weighted', 'finbert_weighted'])
    filename = file.split(".")[0] + "_weighted.csv"
    weighted_sent.to_csv(filename,index=False)
    print(file,"done")

ATER_cleaned_sentiment.csv done
cleaned_AAPL_sentiment.csv done
